## Dwarf Galaxy SFMS Analysis

**Project Goals:**
Investigatehow dwarf-dwarf interactions and environment influence star star formation. I will look at the galaxies that lie close to the SFMS of (Δlog⁡(sSFR)= −0.25 to 0.25 and those above. 

# Objectives
- Load TNG50 galaxy catalog
- Extract sampole of dwarfs
- Construct SFMS
- Calculate Δlog⁡(sSFR)
- Identify galaxy population above SFMS
- Classify galaxies by environment (central, satellite, backsplash) and record population
- Invesigate whether or not dwarf pairs exihit enhanced star formation


First, I will replicate Figure 1 from: _Bhattacharyya, J., Peter, A. H. G., & Leauthaud, A. (2025, April 23). Dwarf galaxies in the TNG50 field: Connecting their star-formation rates with their environments. arXiv.org. https://arxiv.org/abs/2501.01946_ with my own personal markdowns.

#Import Libraries and Configure Plotting Style

This first cell is responisible for importing python libraries required for the visualization and analysis of the dwarf galaxy sample.

In [ ]:
import numpy as np
import numpy.random as npr
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
import matplotlib.colors as mcol
import matplotlib.cm as mcm
import scipy.stats as sts
from matplotlib.lines import Line2D
from matplotlib.legend import Legend
import scipy.optimize as sopt
from sim_read import *

plt.rcParams["axes.linewidth"]  = 2
plt.rcParams["xtick.major.size"]  = 8
plt.rcParams["xtick.minor.size"]  = 3
plt.rcParams["ytick.major.size"]  = 8
plt.rcParams["ytick.minor.size"]  = 3
plt.rcParams["xtick.direction"]  = "in"
plt.rcParams["ytick.direction"]  = "in"
plt.rcParams["legend.frameon"] = 'False'
plt.rcParams.update({'font.size': 24})
plt.rcParams.update({'font.family': 'serif'})
plt.rcParams["mathtext.default"] = 'rm'
plt.rcParams["mathtext.fontset"] = 'cm'

# Loads Simulation Data and Computes Derived Mass Properties

This cell is responsible for intializing the TNG50 simulation data by loading host halo and sub halo catalogs, as well as additional group and galaxy properties. It also computes derived mass quantities.

In [ ]:
gls = sim(exgrp_fields=['GroupFirstSub'],exsub_fields=['SubhaloMass','SubhaloStellarPhotometrics','SubhaloMassType'])
sim.mass_add(gls)

This line creates a selection of host halos that satisfy a minimum mass requirment.

In [ ]:
halo_selc = np.where(gls.hst.group_m200>=10)[0]

##Saves Selected Host Halo Properties to an HDF5 File

This cell is responsible for creating an HDF5 file that contains the properties of host halos that satisfy the minimum halo mass requirment.

In [ ]:
import h5py
hf = h5py.File('tng50_halos_1e10.hdf5', 'w')
 
hf.create_dataset('group_id', data=halo_selc, dtype='int')
hf.create_dataset('group_m200crit', data=gls.hst.group_m200[halo_selc], dtype='float')
hf.create_dataset('group_r200crit', data=gls.hst.Group_R_Crit200[halo_selc]/h0, dtype='float')

print(np.copy(hf.get('group_id')))
print(np.copy(hf.get('group_m200crit')))
print(np.copy(hf.get('group_r200crit')))
print(hf.keys())

hf.close()

##Defines the Galaxy Sample Selection Function

This cell is repossible

In [ ]:
h0 = 0.677
dconv = 1e-3/h0
mconv = 10 - np.log10(h0)

def samp_selc(floor=7.5,thresh=9.5,NN=5,hfloor=10):
    massive_gal,dwarf_gal = dwf_select(gls.sub.mst,thresh,floor)
    all_gal = np.array([*dwarf_gal,*massive_gal])
    
    rh0 = len(all_gal)/50.0**3
    print(len(all_gal),rh0)

    back_host = np.genfromtxt('backsplash.txt',usecols=0,dtype='int')
    back_dz0 = np.genfromtxt('backsplash.txt',usecols=3,dtype='float')
    back_gal = np.genfromtxt('backsplash.txt',usecols=2,dtype='int')

    dhost = [[],[],[]]
    llg,llt,llh,llr = [[],[],[]],[[],[],[]],[[],[],[]],[[],[],[]]
    x200 = [[],[],[]]

    all_gal_pos =  gls.sub.SubhaloPos[all_gal,:]
    massive_gal_pos = gls.sub.SubhaloPos[massive_gal,:] 

    for j in back_gal:
        lh = gls.sub.SubhaloGrNr[j]
        blh = (np.where(j==back_gal)[0])[0]
        
        if gls.hst.group_m200[lh]>=hfloor:
                #dist, ind = massive_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1)
                rall = wrap_dist_1n(massive_gal_pos,gls.sub.SubhaloPos[j,:])
                if gls.sub.mst[j]<9.5: dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]
                else: dist,ind = np.sort(rall)[1:6],np.argsort(rall)[1:6]
                    
                ind = massive_gal[ind]
                dist *= dconv
                MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])
                          
                dhost[2].append(dist[0])
                llt[2].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                llh[2].append(lh)
                llg[2].append(j)
                x200[2].append(back_dz0[blh]/gls.hst.Group_R_Crit200[back_host[blh]])
                    
                #dist0, ind0 = all_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1)
                rall = wrap_dist_1n(all_gal_pos,gls.sub.SubhaloPos[j,:])
                dist0 = np.sort(rall)[:5]
                dist0 *= dconv
                llr[2].append(0.2387324146*(NN+1)/dist0[-1]**3)

        #all_gal = np.delete(all_gal,np.where(all_gal==j)[0])

    all_gal,massive_gal = np.array(list(set(all_gal)-set(back_gal))), np.array(list(set(massive_gal)-set(back_gal)))
    all_gal_tree =  KDTree(gls.sub.SubhaloPos[all_gal,:]) 
    massive_gal_tree = KDTree(gls.sub.SubhaloPos[massive_gal,:]) 
    
    dwarf_host,host_mem = np.unique(gls.sub.SubhaloGrNr[all_gal],return_inverse=True)
    Ndh = len(dwarf_host)
    for i in range(Ndh):  
        lh = dwarf_host[i]
        #print(gls.hst.group_m200[lh])
        if gls.hst.group_m200[lh]>=hfloor:
            dw_mem = all_gal[np.where(host_mem==i)[0]]

            stellar_sort = np.argsort(gls.sub.mst[dw_mem])
            cen = dw_mem[stellar_sort[-1]]

            #r = wrap_dist_1n(gls.sub.SubhaloPos[cen,:],gls.hst.GroupPos[lh,:])
            if len(dw_mem)>=1:
                rall = wrap_dist_1n(massive_gal_pos,gls.sub.SubhaloPos[cen,:])
                if gls.sub.mst[j]<9.5: dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]
                else: dist,ind = np.sort(rall)[1:6],np.argsort(rall)[1:6]                
                ind = massive_gal[ind]  
                dist *= dconv
                MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])

                dhost[0].append(dist[0])
                llt[0].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                llh[0].append(lh)
                llg[0].append(cen)
                x200[0].append(0)
                
                rall =wrap_dist_1n(all_gal_pos,gls.sub.SubhaloPos[cen,:])
                dist0 = np.sort(rall)[:5]
                dist0 *= dconv
                llr[0].append(0.2387324146*(NN+1)/dist0[-1]**3)

                if len(dw_mem)>1:
                    for j in dw_mem[stellar_sort[:-1]]:
                        rall = wrap_dist_1n(massive_gal_pos,gls.sub.SubhaloPos[j,:])
                        if gls.sub.mst[j]<9.5: dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]
                        else: dist,ind = np.sort(rall)[1:6],np.argsort(rall)[1:6] 

                        ind = massive_gal[ind]  
                        dist *= dconv
                        MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])
    
                        dhost[1].append(dist[0])
                        llt[1].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                        llh[1].append(lh)
                        llg[1].append(j)
                        x200[1].append(wrap_dist_0n(gls.sub.SubhaloPos[j,:],gls.hst.GroupPos[lh,:])/gls.hst.Group_R_Crit200[lh])

    
                        rall = wrap_dist_1n(all_gal_pos,gls.sub.SubhaloPos[j,:])
                        dist0 = np.sort(rall)[:5]
                        dist0 *= dconv
                        llr[1].append(0.2387324146*(NN+1)/dist0[-1]**3)
                    
        
    return llt,llh,llg,llr,dhost,x200
